## RAG Appliication usinig TypeSense

In [2]:
import typesense

In [8]:
import os
from dotenv import load_dotenv
import typesense

load_dotenv()

True

In [ ]:
## Configuring the Typesense client
client=typesense.Client({
  'nodes': [{
    'host': 'f6x3bl4zs9nohamep-1.a2.typesense.net',  # For Typesense Cloud use xxx.a1.typesense.net
    'port': '443',       # For Typesense Cloud use 443
    'protocol': 'https'    # For Typesense Cloud use https
  }],
  'api_key':os.getenv("TYPESENSE_API_KEY"),
  'connection_timeout_seconds': 2
})

## Specifying the schema for the collection
books_schema = {
  'name': 'books',
  'fields': [
    {'name': 'title', 'type': 'string'},
    {'name': 'authors', 'type': 'string[]', 'facet': True},
    {'name': 'publication_year', 'type': 'int32', 'facet': True},
    {'name': 'ratings_count', 'type': 'int32'},
    {'name': 'average_rating', 'type': 'float'}
  ],
  'default_sorting_field': 'ratings_count'
}

print(client.collections.create(books_schema))

ObjectAlreadyExists: [Errno 409] A collection with name `books` already exists.

In [ ]:
## Checking if the connection too the client is established or not
client

In [ ]:
## Importing the data into the collection in Typesense
with open('books.jsonl', 'r', encoding='utf-8') as jsonl_file:
    data = jsonl_file.read()
    client.collections['books'].documents.import_(data)

In [ ]:
## Searching for documents in the collection in Typesense
search_parameters={
    'q':"Narnia",
    'query_by':"title,authors",
    'sort_by':"ratings_count:desc"
}

client.collections['books'].documents.search(search_parameters)

{'facet_counts': [],
 'found': 4,
 'hits': [{'document': {'authors': ['C.S. Lewis', ' Pauline Baynes'],
    'average_rating': 4.24,
    'id': '219',
    'image_url': 'https://images.gr-assets.com/books/1449868701m/11127.jpg',
    'publication_year': 1956,
    'ratings_count': 376385,
    'title': 'The Chronicles of Narnia'},
   'highlight': {'title': {'matched_tokens': ['Narnia'],
     'snippet': 'The Chronicles of <mark>Narnia</mark>'}},
   'highlights': [{'field': 'title',
     'matched_tokens': ['Narnia'],
     'snippet': 'The Chronicles of <mark>Narnia</mark>'}],
   'text_match': 578730123365189753,
   'text_match_info': {'best_field_score': '1108091338753',
    'best_field_weight': 15,
    'fields_matched': 1,
    'num_tokens_dropped': 0,
    'score': '578730123365189753',
    'tokens_matched': 1,
    'typo_prefix_score': 0}},
  {'document': {'authors': ['C.S. Lewis'],
    'average_rating': 3.96,
    'id': '347',
    'image_url': 'https://images.gr-assets.com/books/1308814880m/121

## Langchain + Typsense + Groq LLM + RAG Application

In [16]:
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import Typesense
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_groq import ChatGroq

In [17]:
loader = TextLoader("test.txt")
documents = loader.load()
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
docs = text_splitter.split_documents(documents)

embeddings = HuggingFaceEmbeddings()

/var/folders/3d/ttrr9m652ys7prc8v11b1j200000gn/T/ipykernel_28532/13807947.py:6: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings()
/var/folders/3d/ttrr9m652ys7prc8v11b1j200000gn/T/ipykernel_28532/13807947.py:6: LangChainDeprecationWarning: Default values for HuggingFaceEmbeddings.model_name were deprecated in LangChain 0.2.16 and will be removed in 0.4.0. Explicitly pass a model_name to the HuggingFaceEmbeddings constructor instead.
  embeddings = HuggingFaceEmbeddings()


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [18]:
## Creating a vector store using Typesense and adding the documents to it
docsearch=Typesense.from_documents(
    docs,
    embeddings,
    typesense_client_params={
        "host": "f6x3bl4zs9nohamep-1.a2.typesense.net",  # Use xxx.a1.typesense.net for Typesense Cloud
        "port": "443",  # Use 443 for Typesense Cloud
        "protocol": "https",  # Use https for Typesense Cloud
        "typesense_api_key":os.getenv("TYPESENSE_API_KEY"),
        "typesense_collection_name": "lang-chain"
    },
)

In [19]:
query = "What are the applications of artificial intelligence?"
found_docs = docsearch.similarity_search(query)
print(found_docs[0].page_content)

Artificial Intelligence (AI) - Comprehensive Guide

What is AI?
Artificial Intelligence (AI) is the field of computer science focused on creating systems that can perform tasks requiring human intelligence. These tasks include learning, reasoning, decision-making, perception, and language understanding.

Brief History of AI:
- 1950: Alan Turing proposes the Turing Test
- 1956: Dartmouth Conference (birth of AI as a field)
- 1970s–80s: Development of expert systems
- 1997: IBM's Deep Blue defeats world chess champion Garry Kasparov
- 2010s: Rise of deep learning and big data
- 2020s: Growth of generative AI and large language models

Branches of AI:
1. Machine Learning (ML)
   - Systems learn from data
   - Includes supervised, unsupervised, and reinforcement learning

2. Deep Learning (DL)
   - Uses artificial neural networks with many layers
   - Powers image recognition and speech processing


In [20]:
### Retriever
retriever = docsearch.as_retriever()
retriever

VectorStoreRetriever(tags=['Typesense', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.typesense.Typesense object at 0x1290b5f70>, search_kwargs={})

In [22]:
query = "Artificial intelligence applications?"
retriever.invoke(query)[0]

Document(metadata={'source': 'test.txt'}, page_content="Artificial Intelligence (AI) - Comprehensive Guide\n\nWhat is AI?\nArtificial Intelligence (AI) is the field of computer science focused on creating systems that can perform tasks requiring human intelligence. These tasks include learning, reasoning, decision-making, perception, and language understanding.\n\nBrief History of AI:\n- 1950: Alan Turing proposes the Turing Test\n- 1956: Dartmouth Conference (birth of AI as a field)\n- 1970s–80s: Development of expert systems\n- 1997: IBM's Deep Blue defeats world chess champion Garry Kasparov\n- 2010s: Rise of deep learning and big data\n- 2020s: Growth of generative AI and large language models\n\nBranches of AI:\n1. Machine Learning (ML)\n   - Systems learn from data\n   - Includes supervised, unsupervised, and reinforcement learning\n\n2. Deep Learning (DL)\n   - Uses artificial neural networks with many layers\n   - Powers image recognition and speech processing")